In [0]:
from pyspark.sql import functions as F

df_gold = spark.read.table("gold_credit_features")

import pandas as pd
import numpy as np

np.random.seed(99)
n = 10000

df_new = pd.DataFrame({
    'age':                np.random.randint(20, 45, n),      
    'income':             np.random.randint(20000, 80000, n),
    'loan_amount':        np.random.randint(5000, 500000, n),
    'debt_ratio':         np.round(np.random.uniform(0.5, 0.9, n), 2), 
    'credit_score':       np.random.randint(300, 600, n), 
    'num_late_payments':  np.random.randint(0, 20, n),
    'employment_years':   np.random.randint(0, 40, n),
    'monthly_expenses':   np.random.randint(1000, 20000, n),
    'savings_balance':    np.random.randint(0, 500000, n),
    'credit_utilization': np.round(np.random.uniform(0.0, 1.0, n), 2),
    'num_credit_lines':   np.random.randint(1, 15, n),
    'num_dependents':     np.random.randint(0, 5, n),
    'previous_default':   np.random.randint(0, 2, n),
})

df_new['default'] = (
    (df_new['debt_ratio'] > 0.6) &
    (df_new['credit_score'] < 500) &
    (df_new['num_late_payments'] > 5)
).astype(int)

df_new['loan_to_income_ratio'] = round(df_new['loan_amount'] / df_new['income'], 4)
df_new['expense_to_income_ratio'] = round(df_new['monthly_expenses'] * 12 / df_new['income'], 4)
df_new['savings_to_loan_ratio'] = round(df_new['savings_balance'] / df_new['loan_amount'], 4)
df_new['financial_stress_score'] = round(
    df_new['debt_ratio'] * 0.4 +
    df_new['credit_utilization'] * 0.3 +
    (df_new['num_late_payments'] / 20) * 0.3, 4
)

df_new_spark = spark.createDataFrame(df_new)

In [0]:
from pyspark.sql import functions as F

monitor_cols = ['age', 'income', 'credit_score', 'debt_ratio', 'credit_utilization']

baseline_exprs = []
for c in monitor_cols:
    baseline_exprs.append(F.mean(c).alias(f"{c}_mean"))
    baseline_exprs.append(F.stddev(c).alias(f"{c}_std"))

baseline_stats = df_gold.select(baseline_exprs).collect()[0]

new_exprs = []
for c in monitor_cols:
    new_exprs.append(F.mean(c).alias(f"{c}_mean"))
    new_exprs.append(F.stddev(c).alias(f"{c}_std"))

new_stats = df_new_spark.select(new_exprs).collect()[0]

drift_detected = False

for col in monitor_cols:
    baseline_mean = baseline_stats[f"{col}_mean"]
    new_mean = new_stats[f"{col}_mean"]
    change_pct = abs(new_mean - baseline_mean) / baseline_mean * 100
    
    status = "DRIFT" if change_pct > 10 else "normal"
    print(f"{col:25s} base: {baseline_mean:8.2f}  new data: {new_mean:8.2f}  changed: {change_pct:5.1f}%  {status}")
    
    if change_pct > 10:
        drift_detected = True


In [0]:
from datetime import date

drift_records = []
for col in monitor_cols:
    baseline_mean = baseline_stats[f"{col}_mean"]
    new_mean = new_stats[f"{col}_mean"]
    change_pct = abs(new_mean - baseline_mean) / baseline_mean * 100
    
    drift_records.append({
        "feature": col,
        "baseline_mean": float(baseline_mean),
        "new_mean": float(new_mean),
        "change_pct": float(change_pct),
        "drift_detected": change_pct > 10,
        "monitor_date": str(date.today())
    })

import pandas as pd
df_drift = spark.createDataFrame(pd.DataFrame(drift_records))

df_drift.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("monitoring_drift_log")

display(df_drift)